In [4]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("Data/python.txt",encoding="utf-8")

C:\Users\vk952\AppData\Local\Temp\ipykernel_17024\1132379991.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [5]:
document = loader.load()

In [6]:
document

[Document(metadata={'source': 'Data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [7]:
# pdf data
from langchain_community.document_loaders.pdf import PyPDFLoader
pdf_loader = PyPDFLoader("Data/NIPS-2017-attention-is-all-you-need-Paper.pdf")

document = pdf_loader.load()
document

[Document(metadata={'producer': 'pdfcpu v0.15.0 dev', 'creator': 'PyPDF', 'creationdate': '2026-11-03T16:46:31+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

## Ingestion Pipeline

In [8]:
# data => Documenta
import os
from langchain_community.document_loaders import PyPDFLoader

In [9]:


def load_all_pdfs():
    folder_path = "Data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".pdf"):

            # Complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("Total PDFs:", num_docs)
    print("Total pages:", len(all_docs))

    return all_docs


In [10]:
all_pdf_documents = load_all_pdfs()

Total PDFs: 2
Total pages: 26


In [11]:
# chunks
# !pip install langchain_text_splitters


In [14]:
from  langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)

    return chunked_docs

In [15]:
chunks = split_docs(all_pdf_documents)

In [17]:
len(chunks)

276

## Embeding

In [18]:
from sentence_transformers import SentenceTransformer

W1109 17:17:08.629000 17024 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [98]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model  ....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape : ", embeddings.shape)
        return embeddings

In [99]:
embeddings_manager = EmbeddingManager()

loading model  .... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions =  384


C:\Users\vk952\AppData\Local\Temp\ipykernel_17024\4147707852.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())


In [100]:
import chromadb
import uuid
import os

### vector store 

In [101]:

import os
import uuid
import chromadb

class VectorStoreManager:
    def __init__(self, persist_directory="Data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create a collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):

        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # store => ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # add documents to collection
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in collection =", len(documents_content))
        print("docs in collection:", self.collection.count())



In [102]:
vector_store = VectorStoreManager()

initialized the vector store with collection pdf_documents
docs in collection: 276


In [103]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embeddings_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

embeddings shape :  (276, 384)
total documents added in collection = 276
docs in collection: 552


## Retrival pipeline


In [104]:
from sklearn.metrics.pairwise import cosine_similarity

In [105]:

class RAGRetriever:
    def __init__(self, embeddings_manager, vector_store):
        self.embeddings_manager = embeddings_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):

        # query => embedding
        query_embeddings = self.embeddings_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs = []

        if results["documents"] and results["documents"][0]:

            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):

                similarity_score = 1 - distance

                if similarity_score >= score_threshold:

                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs



In [106]:
rag_retriever = RAGRetriever(embeddings_manager, vector_store)

In [108]:
rag_retriever.retrieve("what is  Modular RAG ")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape :  (1, 384)
retrieved 5 documents


[{'id': 'doc_abf3f35b-232e-46b5-b5c9-2220ec9f6d25',
  'document': 'vations like restructured RAG modules [13] and rearranged\nRAG pipelines [14] have been introduced to tackle specific\nchallenges. The shift towards a modular RAG approach is\nbecoming prevalent, supporting both sequential processing and\nintegrated end-to-end training across its components. Despite\nits distinctiveness, Modular RAG builds upon the foundational\nprinciples of Advanced and Naive RAG, illustrating a progres-\nsion and refinement within the RAG family.',
  'metadata': {'creator': 'LaTeX with hyperref',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25',
   'total_pages': 21,
   'source': 'Data/pdfs\\research2.pdf',
   'author': '',
   'doc_index': 65,
   'page': 3,
   'title': '',
   'content_length': 463,
   'page_label': '4',
   'subject': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'trapped': '/False'

## Integrate with LLMs

## openAI - gpt

In [109]:
import os
API_KEY_OPENAI = os.environ["OPENAI_API_KEY"]


In [110]:
!pip install langchain-openai

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 2.4 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 2.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 2.5 MB/s eta 0:00:01
   --------------------- ------------------ 1.3/2.4 MB 1.4 MB/s eta 0:00:01
   ------------------------- -------------- 1.6/2.4 MB 1.5 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 1.5 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 1.4 MB/s  0:00:01
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/940.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/940.9 kB 2.3 MB/s eta 0:00:01
   ------------------------------

In [116]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [117]:
# generate our retrievel-augmented output
def generate_output(query, retriever,llm,top_k=3):
    results = retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for  doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f"""use given context to generate the answer for the query
                  Context:{context}
                  Query:{query}   """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [118]:
answer = generate_output("what is RAG ?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape :  (1, 384)
retrieved 3 documents


OpenAIRateLimitError: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.

In [120]:
answer

NameError: name 'answer' is not defined

In [119]:
answer

NameError: name 'answer' is not defined